## Chapter 3: Feature Engineering — Understanding Customer Behavior

  ### Why Feature Engineering?
  Raw transaction data tells us WHAT happened — customer 17850 bought
  6 units of a product on December 1st 2010 for £15.30. But our models
  don't learn from individual transactions. They learn from customer
  behavior patterns over time.

  Feature Engineering is the process of transforming raw transactions
  into meaningful customer-level signals that capture behavior patterns.

  ### RFM Analysis — The Gold Standard
  The most proven framework for understanding customer behavior is RFM:

  - **Recency (R):** How recently did the customer buy?
    A customer who bought yesterday is very different from one who
    bought 6 months ago. Recent buyers are more likely to buy again.

  - **Frequency (F):** How often do they buy?
    A customer who buys every week is far more valuable and less likely
    to churn than one who bought only once.

  - **Monetary (M):** How much do they spend?
    High spenders represent your most valuable customers — losing them
    has a bigger business impact than losing low spenders.

  These 3 numbers alone can tell you almost everything about a customer's
  relationship with your business.

In [264]:
'''
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
  # Load cleaned data
df = pd.read_csv('/Users/purvamugdiya/Desktop/customer-intelligence-platform/data/cleaned_retail.csv', parse_dates=['InvoiceDate'])
df.head()
# Reference date — one day after the last transaction in the dataset
# This is our "today" from the business's perspective
reference_date = df['InvoiceDate'].max() + pd.Timedelta(days=1)
print("Reference date:", reference_date)
# Build RFM features at customer level
rfm = df.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalAmount', 'sum')
).reset_index()
print("\nRFM Table Shape:", rfm.shape)
print("\nSample RFM:")
rfm.head(10)
  # Statistical summary of RFM
print("RFM Summary Statistics:")
print(rfm[['Recency', 'Frequency', 'Monetary']].describe().round(2))
# Visualize distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(rfm['Recency'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Recency Distribution')
axes[0].set_xlabel('Days since last purchase')
axes[0].set_ylabel('Number of customers')

axes[1].hist(rfm['Frequency'], bins=50, color='seagreen', edgecolor='white')
axes[1].set_title('Frequency Distribution')
axes[1].set_xlabel('Number of purchases')
axes[1].set_ylabel('Number of customers')

axes[2].hist(rfm['Monetary'], bins=50, color='coral', edgecolor='white')
axes[2].set_title('Monetary Distribution')
axes[2].set_xlabel('Total spend (GBP)')
axes[2].set_ylabel('Number of customers')

plt.tight_layout()
plt.savefig('../data/rfm_distributions.png')
plt.show()
rfm.to_csv('../data/rfm.csv', index=False)
print("RFM saved!")
# Define churn threshold
CHURN_THRESHOLD = 90  # days
# Create churn label
# 1 = churned (hasn't bought in 90+ days)
# 0 = active (bought within last 90 days)
rfm['Churn'] = (rfm['Recency'] >= CHURN_THRESHOLD).astype(int)
# Check churn distribution
churn_counts = rfm['Churn'].value_counts()
churn_pct = rfm['Churn'].value_counts(normalize=True) * 100
print("Churn Distribution:")
print(f"Active customers (0):  {churn_counts[0]} ({churn_pct[0]:.1f}%)")
print(f"Churned customers (1): {churn_counts[1]} ({churn_pct[1]:.1f}%)")
# Visualize
plt.figure(figsize=(6, 4))
plt.bar(['Active', 'Churned'],
        [churn_counts[0], churn_counts[1]],
        color=['seagreen', 'coral'])
plt.title('Customer Churn Distribution')
plt.ylabel('Number of Customers')
for i, v in enumerate([churn_counts[0], churn_counts[1]]):
    plt.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../data/churn_distribution.png')
plt.show()
 # Save RFM with churn labels
rfm.to_csv('../data/rfm.csv', index=False)
print("Saved! Shape:", rfm.shape)
print("Columns:", rfm.columns.tolist())'''


'\nimport pandas as pd\nimport numpy as np\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n  # Load cleaned data\ndf = pd.read_csv(\'/Users/purvamugdiya/Desktop/customer-intelligence-platform/data/cleaned_retail.csv\', parse_dates=[\'InvoiceDate\'])\ndf.head()\n# Reference date — one day after the last transaction in the dataset\n# This is our "today" from the business\'s perspective\nreference_date = df[\'InvoiceDate\'].max() + pd.Timedelta(days=1)\nprint("Reference date:", reference_date)\n# Build RFM features at customer level\nrfm = df.groupby(\'CustomerID\').agg(\n    Recency   = (\'InvoiceDate\', lambda x: (reference_date - x.max()).days),\n    Frequency = (\'InvoiceNo\',   \'nunique\'),\n    Monetary  = (\'TotalAmount\', \'sum\')\n).reset_index()\nprint("\nRFM Table Shape:", rfm.shape)\nprint("\nSample RFM:")\nrfm.head(10)\n  # Statistical summary of RFM\nprint("RFM Summary Statistics:")\nprint(rfm[[\'Recency\', \'Frequency\', \'Monetary\']].describe().round(2))\n# Visu

In [265]:
'''Understanding the RFM Output

  Each row now represents ONE customer — not one transaction.
  We have collapsed hundreds of thousands of transactions into
  a single behavioral fingerprint per customer.

  Let's look at what these numbers mean:
  - A customer with Recency = 5 bought just 5 days ago — very engaged
  - A customer with Recency = 300 hasn't bought in 300 days — at risk of churning
  - A customer with Frequency = 1 has only bought once — one-time buyer
  - A customer with Frequency = 50 is a loyal repeat customer
  - A customer with Monetary = 50 is a low spender
  - A customer with Monetary = 5000 is a high value customer'''

'''Interpreting the RFM Output
  
  We have 4,338 unique customers — each now has a behavioral fingerprint.

  ---
  Recency (avg: 92 days, max: 374 days)
  - The average customer last bought ~3 months ago
  - 25% of customers bought within the last 18 days — these are your most engaged buyers
  - 25% of customers haven't bought in 142+ days — these are at serious churn risk
  - Some customers haven't bought in 374 days — almost certainly churned
  
  Frequency (avg: 4.27, max: 209)
  - Half the customers bought only 1-2 times — mostly one-time buyers
  - A small group bought 50-209 times — these are your loyal core customers
  - This confirms the 80/20 rule — a small group drives most of the business

  Monetary (avg: £2,054, max: £280,206)
  - Huge variation — most customers spend modestly but a few spend enormously
  - Customer 12346 is fascinating — bought only ONCE, spent £77,183, hasn't returned in 326 days — a high value churned customer,
  exactly the type a business desperately wants back

  ---
  So What?
  
  These numbers give us everything we need to predict churn. The pattern is clear — customers with high Recency + low Frequency are
   your biggest churn risk.'''

'''## Creating the Churn Label

  ### What is Churn?
  Churn means a customer has stopped doing business with you. But unlike
  a subscription service where a customer explicitly cancels, in retail
  churn is silent — customers simply stop buying without telling you.

  So how do we define churn from transaction data?

  ### Our Churn Definition
  We use Recency as our churn indicator:
  - If a customer hasn't bought in more than 90 days → Churned (1)
  - If a customer bought within the last 90 days → Active (0)

  ### Why 90 days?
  Looking at our RFM statistics:
  - The average Recency is 92 days
  - The 75th percentile is 142 days

  90 days (3 months) is a widely used industry standard for retail churn.
  It means if a customer hasn't returned in a quarter, they are considered
  at risk of never coming back. This is a business decision — different
  industries use different thresholds (telecom uses 30 days, retail typically
  uses 60-90 days).'''

'''This tells us the split between active and churned customers in our dataset.

A healthy churn analysis needs both classes represented — if 99% of
customers were active, our model would just predict "active" for everyone
and still be 99% accurate, which is useless.
### So What?
We now have our target variable — the thing our model will learn to predict.
Every customer is now labeled as either churned or active based on their
buying behavior. In Step 4 we will train XGBoost to predict this label
using their RFM features.'''

'''Interpreting the Churn Distribution
  
2,880 Active customers (66.4%)
These customers bought within the last 90 days — they are engaged and returning.
1,458 Churned customers (33.6%)
1 in 3 customers has not returned in over 90 days. For a real business this is a significant problem — losing a third of your customer base is extremely costly.

Why this distribution is good for modeling:
Our dataset is reasonably balanced — not perfectly 50/50 but not severely imbalanced either. This means our XGBoost model will have enough examples of both churned and active customers to learn the difference reliably. No special resampling techniques
needed.'''

'Interpreting the Churn Distribution\n\n2,880 Active customers (66.4%)\nThese customers bought within the last 90 days — they are engaged and returning.\n1,458 Churned customers (33.6%)\n1 in 3 customers has not returned in over 90 days. For a real business this is a significant problem — losing a third of your customer base is extremely costly.\n\nWhy this distribution is good for modeling:\nOur dataset is reasonably balanced — not perfectly 50/50 but not severely imbalanced either. This means our XGBoost model will have enough examples of both churned and active customers to learn the difference reliably. No special resampling techniques\nneeded.'

## Fixing Data Leakage — Time-Based Churn Definition
### The Right Approach
Instead of defining churn from the same Recency we use as a feature,
we split the dataset into two time periods:
- **Observation Period (Dec 2010 – Sep 2011):**
  We calculate RFM features from this period — what did the
  customer do in the past?
- **Prediction Period (Oct 2011 – Dec 2011):**
  We check if the customer bought during this period — did they
  come back or not?
A customer who bought in the observation period but NOT in the
prediction period is defined as churned.
This mirrors exactly how a real business would build this model —
train on historical behavior, predict future behavior.

In [305]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [306]:
# Load cleaned data
df = pd.read_csv('../data/cleaned_retail.csv', parse_dates=['InvoiceDate'])
df['TotalAmount'] = df['Quantity'] * df['UnitPrice']

In [307]:
# Define time periods
observation_end   = pd.Timestamp('2011-09-30')
prediction_start  = pd.Timestamp('2011-10-01')
prediction_end    = pd.Timestamp('2011-12-09')

In [310]:
# Split into observation and prediction periods
obs  = df[df['InvoiceDate'] <= observation_end]
pred = df[(df['InvoiceDate'] >= prediction_start) &
          (df['InvoiceDate'] <= prediction_end)]
print(f"Observation period transactions:  {len(obs):,}")
print(f"Prediction period transactions:   {len(pred):,}")

Observation period transactions:  265,042
Prediction period transactions:   130,772


In [311]:
# Only keep customers who appear in observation period
obs_customers  = set(obs['CustomerID'].unique())
pred_customers = set(pred['CustomerID'].unique())
print(f"\nCustomers in observation period:  {len(obs_customers):,}")
print(f"Customers in prediction period:   {len(pred_customers):,}")


Customers in observation period:  3,604
Customers in prediction period:   2,556


In [313]:
# Calculate RFM on observation period only
reference_date = observation_end + pd.Timedelta(days=1)
rfm = obs.groupby('CustomerID').agg(
    Recency   = ('InvoiceDate', lambda x: (reference_date - x.max()).days),
    Frequency = ('InvoiceNo',   'nunique'),
    Monetary  = ('TotalAmount', 'sum')
).reset_index()

In [314]:
# Define churn label from prediction period
# Churned = 1 if customer did NOT buy in prediction period
# Active  = 0 if customer DID buy in prediction period
rfm['Churn'] = (~rfm['CustomerID'].isin(pred_customers)).astype(int)
print("RFM Shape:", rfm.shape)
print("\nChurn Distribution:")
churn_counts = rfm['Churn'].value_counts()
churn_pct    = rfm['Churn'].value_counts(normalize=True) * 100
print(f"Active  (0): {churn_counts[0]} ({churn_pct[0]:.1f}%)")
print(f"Churned (1): {churn_counts[1]} ({churn_pct[1]:.1f}%)")

RFM Shape: (3604, 5)

Churn Distribution:
Active  (0): 1831 (50.8%)
Churned (1): 1773 (49.2%)


### Why This is Better

| Old Approach | New Approach |
|---|---|
| Churn defined using Recency | Churn defined using future behavior |
| Model reads the answer from features | Model genuinely predicts future |
| ROC-AUC = 0.9999 (fake) | ROC-AUC will be realistic |
| Data leakage | No leakage |
This is how churn models are built in production at real companies.
The model now has a genuine challenge — predict who will come back
based only on what they did in the past.


In [315]:
# Save corrected RFM
rfm.to_csv('../data/rfm.csv', index=False)
print("Saved successfully!")
print("Columns:", rfm.columns.tolist())

Saved successfully!
Columns: ['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Churn']


Interpreting the New Distribution
  
  3,604 customers in observation period
  These are the customers whose past behavior we'll use to train the model.

  2,556 customers returned in prediction period
  Meaning 1,048 customers who bought in the first 9 months never came back in the last 3 months — those are our churned customers.

  50.8% Active vs 49.2% Churned
  This is nearly perfectly balanced — the best possible distribution for a classification model. No class imbalance issues at all.

  Compare to before:
  - Old approach: ROC-AUC 0.9999 — fake, due to data leakage
  - New approach: Model will need to genuinely learn patterns — results will be realistic and defensible in interviews

  ---
  So What?
  
  Half your customers stopped returning after 9 months. That's a massive business problem — and exactly why a churn prediction model is valuable. If the business can identify which customers are about to leave, they can intervene before it's too late.

## Adding Richer Features

  Basic RFM gives us 3 signals. We add 3 more to give the model a fuller picture of each customer:

  - **AvgOrderValue** — separates high-spending-per-visit customers from frequent small buyers
  - **PurchaseSpread** — how many days between their first and last purchase in the dataset.
    Long-tenure customers have invested in the relationship and are less likely to churn.
  - **UniqueProducts** — how many distinct products they bought.
    Customers who explore widely are more engaged than repeat single-item buyers.

In [316]:
# Compute additional features from the same feature window
extra = obs.groupby('CustomerID').agg(
    AvgOrderValue  = ('TotalAmount', 'mean'),
    PurchaseSpread = ('InvoiceDate', lambda x: (x.max() - x.min()).days),
    UniqueProducts = ('StockCode',   'nunique')
).reset_index()

We're at 72.76% — inching up but plateauing. The model has extracted nearly everything it can from the current features. We need to think differently.

  Why we're stuck: Every feature we have is a summary statistic — totals and averages across the full 9-month window. The model has no idea if a customer was buying frequently in January but stopped by August. That declining trend is actually the
  strongest churn signal, and we're completely blind to it.

  What breaks the ceiling: Quarterly trend features.

In [317]:
# Split feature window into quarters to capture behavioral trend
q1 = obs[(obs['InvoiceDate'] >= '2011-01-01') & (obs['InvoiceDate'] <= '2011-03-31')]
q2 = obs[(obs['InvoiceDate'] >= '2011-04-01') & (obs['InvoiceDate'] <= '2011-06-30')]
q3 = obs[(obs['InvoiceDate'] >= '2011-07-01') & (obs['InvoiceDate'] <= '2011-09-30')]

In [318]:
q1_orders = q1.groupby('CustomerID')['InvoiceNo'].nunique().rename('Q1_Orders')
q2_orders = q2.groupby('CustomerID')['InvoiceNo'].nunique().rename('Q2_Orders')
q3_orders = q3.groupby('CustomerID')['InvoiceNo'].nunique().rename('Q3_Orders')

In [319]:
rfm = rfm.merge(q1_orders, on='CustomerID', how='left')
rfm = rfm.merge(q2_orders, on='CustomerID', how='left')
rfm = rfm.merge(q3_orders, on='CustomerID', how='left')
rfm[['Q1_Orders', 'Q2_Orders', 'Q3_Orders']] = rfm[['Q1_Orders', 'Q2_Orders', 'Q3_Orders']].fillna(0)

In [320]:
# Trend: positive = customer is engaging more, negative = disengaging (churn risk)
rfm['PurchaseTrend'] = rfm['Q3_Orders'] - rfm['Q1_Orders']

In [321]:
# Was the customer active in the last quarter of the feature window?
rfm['ActiveInQ3'] = (rfm['Q3_Orders'] > 0).astype(int)

In [322]:
print(rfm[['Q1_Orders', 'Q2_Orders', 'Q3_Orders', 'PurchaseTrend', 'ActiveInQ3']].describe().round(2))

       Q1_Orders  Q2_Orders  Q3_Orders  PurchaseTrend  ActiveInQ3
count    3604.00    3604.00    3604.00        3604.00     3604.00
mean        0.90       1.12       1.19           0.29        0.59
std         1.71       2.08       2.12           1.61        0.49
min         0.00       0.00       0.00          -6.00        0.00
25%         0.00       0.00       0.00          -1.00        0.00
50%         0.00       1.00       1.00           0.00        1.00
75%         1.00       1.00       1.00           1.00        1.00
max        26.00      45.00      54.00          30.00        1.00


The intuition behind this: A customer who bought 5 times in Q1 but 0 times in Q3 is clearly disengaging — PurchaseTrend = -5. A customer who bought in Q3 (ActiveInQ3 = 1) is much less likely to churn than one who went silent in Q2. These two features
  alone should move ROC-AUC significantly.

In [323]:
# Merge into RFM table
rfm = rfm.merge(extra, on='CustomerID', how='left')
rfm['PurchaseSpread'] = rfm['PurchaseSpread'].fillna(0)

In [282]:
rfm.head()

,CustomerID,Recency,Frequency,Monetary,Churn,LogMonetary,IsOneTimeBuyer,AvgOrderValue,PurchaseSpread,UniqueProducts,LogAvgOrderValue
0,12346,255,1,77183.60,1,11.253955,1,77183.600000,0,1,11.253955
1,12347,59,5,2790.86,0,7.934463,0,22.506935,237,82,3.157296
2,12348,5,4,1797.24,1,7.494564,0,57.975484,282,22,4.077122
3,12350,240,1,334.40,1,5.815324,1,19.670588,0,17,3.028712
4,12352,2,7,2194.31,0,7.694079,0,31.347286,224,47,3.476530


####Changes after the ROC-AUC is improved but still at 72%, so further working on Outliers. 

When mean >> median and the max is an extreme outlier, that's the signature of right-skew. The interpretation cell in notebook 03 even called it out explicitly — "Also right skewed — most customers spend modest amounts while a small group of high value
   customers spend significantly more."

  We also saw a specific customer: CustomerID 12346 — bought once, spent £77,183. That single data point pulls the distribution's tail far to the right.

  2. It's a known pattern in retail data. Spending data is almost universally right-skewed — a small group of high-value customers spends orders of magnitude more than the average. This is the same 80/20 rule we discussed in the RFM analysis. It's so
  consistent in retail datasets that you can assume it before even looking.

  ---
  Why does skew hurt the model?
  
  XGBoost splits features at thresholds. When most customers cluster between £300–£2,000 and a few outliers sit at £50,000–£280,000, the model wastes splits trying to handle those extremes. Log transformation compresses the scale so the distribution
  looks roughly bell-shaped — the splits become more meaningful and the model learns better from the majority of customers rather than being distracted by outliers.

In [324]:
# Log-transform skewed monetary features
# Skewed distributions confuse tree splits — log scale fixes this
rfm['LogMonetary']      = np.log1p(rfm['Monetary'])
rfm['LogAvgOrderValue'] = np.log1p(rfm['AvgOrderValue'])

In [325]:
# One-time buyer flag — strongest single churn predictor in retail
# A customer who only ever bought once behaves completely differently
rfm['IsOneTimeBuyer'] = (rfm['Frequency'] == 1).astype(int)


In [326]:
# Average days between purchases (regularity signal)
# A customer who buys every 2 weeks is very different from one who buys randomly
rfm['AvgDaysBetweenPurchases'] = rfm.apply(
    lambda row: row['PurchaseSpread'] / (row['Frequency'] - 1) if row['Frequency'] > 1 else 0,
    axis=1
)

In [327]:
print('Final RFM shape:', rfm.shape)
print('Columns:', rfm.columns.tolist())

Final RFM shape: (3604, 17)
Columns: ['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Churn', 'Q1_Orders', 'Q2_Orders', 'Q3_Orders', 'PurchaseTrend', 'ActiveInQ3', 'AvgOrderValue', 'PurchaseSpread', 'UniqueProducts', 'LogMonetary', 'LogAvgOrderValue', 'IsOneTimeBuyer', 'AvgDaysBetweenPurchases']


In [285]:
rfm.head()

,CustomerID,Recency,Frequency,Monetary,Churn,LogMonetary,IsOneTimeBuyer,AvgOrderValue,PurchaseSpread,UniqueProducts,LogAvgOrderValue,AvgDaysBetweenPurchases
0,12346,255,1,77183.60,1,11.253955,1,77183.600000,0,1,11.253955,0.000000
1,12347,59,5,2790.86,0,7.934463,0,22.506935,237,82,3.157296,59.250000
2,12348,5,4,1797.24,1,7.494564,0,57.975484,282,22,4.077122,94.000000
3,12350,240,1,334.40,1,5.815324,1,19.670588,0,17,3.028712,0.000000
4,12352,2,7,2194.31,0,7.694079,0,31.347286,224,47,3.476530,37.333333


In [328]:
print('Enriched RFM shape:', rfm.shape)
print('Columns:', rfm.columns.tolist())
print()

Enriched RFM shape: (3604, 17)
Columns: ['CustomerID', 'Recency', 'Frequency', 'Monetary', 'Churn', 'Q1_Orders', 'Q2_Orders', 'Q3_Orders', 'PurchaseTrend', 'ActiveInQ3', 'AvgOrderValue', 'PurchaseSpread', 'UniqueProducts', 'LogMonetary', 'LogAvgOrderValue', 'IsOneTimeBuyer', 'AvgDaysBetweenPurchases']



In [329]:
print(rfm[['Recency','Frequency','Monetary','AvgOrderValue','PurchaseSpread','UniqueProducts', 'LogAvgOrderValue', 'AvgDaysBetweenPurchases']].describe().round(2))

       Recency  Frequency   Monetary  AvgOrderValue  PurchaseSpread  \
count  3604.00    3604.00    3604.00        3604.00         3604.00   
mean     93.02       3.63    1707.00          60.11           97.05   
std      86.56       6.08    6978.68        1309.58          104.01   
min       1.00       1.00       2.90           1.45            0.00   
25%      19.00       1.00     284.03          12.92            0.00   
50%      64.00       2.00     598.60          17.83           63.00   
75%     148.00       4.00    1431.95          25.50          190.00   
max     303.00     137.00  203106.64       77183.60          302.00   

       UniqueProducts  LogAvgOrderValue  AvgDaysBetweenPurchases  
count         3604.00           3604.00                  3604.00  
mean            52.12              3.02                    38.31  
std             71.92              0.84                    50.30  
min              1.00              0.90                     0.00  
25%             14.00    

In [330]:
rfm.to_csv('../data/rfm.csv', index=False)
print('Saved rfm.csv with', rfm.shape[1], 'columns and', rfm.shape[0], 'customers')

Saved rfm.csv with 17 columns and 3604 customers
